# 2. Relecture IA des exigences

Le modèle évalue chaque exigence selon les critères de l'**ISO 26262-8 §6.4.2.4**
(non ambiguë, compréhensible, atomique, cohérente, faisable, vérifiable, nécessaire)
et propose une reformulation.

> ⚠️ Les résultats sont des **propositions à valider** par un ingénieur sûreté.

In [1]:
# Rend le package safety_agent importable depuis le dossier notebooks/
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
EXAMPLES = ROOT / "examples"

import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
# Configuration du modèle hébergé (Ollama par défaut)
BASE_URL = "http://localhost:11434/v1"   # vLLM : "http://serveur:8000/v1"
MODEL = "qwen2.5:7b"     # qwen2.5:14b ou plus si vous avez une carte graphique

from safety_agent.notebook import get_llm
llm = get_llm(BASE_URL, MODEL)   # bascule en MODE DÉMO si le serveur ne répond pas

⚠️  Serveur LLM injoignable à http://localhost:11434/v1 (<urlopen error [Errno 111] Connection refused>)
➡️  MODE DÉMO : réponses simulées, sans IA. Lancez Ollama/vLLM pour de vraies analyses.


In [3]:
from safety_agent.loaders import load_project

# Remplacez par vos fichiers (.csv ou .xlsx)
REQS = EXAMPLES / "bms_exigences.csv"
FMEA = EXAMPLES / "bms_amdec.csv"

project, load_findings = load_project(REQS, FMEA)
print(f"{len(project.requirements)} exigences, {len(project.fmea)} lignes AMDEC")

11 exigences, 6 lignes AMDEC


## Relire une sélection d'exigences
Laissez `IDS = None` pour tout relire (plus long).

In [4]:
from safety_agent.ai import review_requirements
from safety_agent.notebook import findings_df, proposals_df

IDS = ["FSR-02", "FSR-06"]
ai_findings, proposals = review_requirements(llm, project, IDS,
                                             on_progress=lambda n, t, rid: print(f"[{n}/{t}] {rid}"))

[1/2] FSR-02
[2/2] FSR-06


In [5]:
findings_df(ai_findings)

,Code,Niveau,Élément,Constat,Source
0,IA-AMBIG,warning,FSR-06,"Exigence ambiguë : termes non définis : appropriée, rapidement",IA
1,IA-ATOM,warning,FSR-06,Exigence non atomique : plusieurs obligations dans une même exigence,IA
2,IA-VERIF,warning,FSR-06,Exigence non vérifiable : pas de critère de réussite chiffré,IA


In [6]:
proposals_df(proposals)

,Type,Cible,Proposition,Justification,Statut
0,reformulation,FSR-06,[DÉMO] Reformuler avec une valeur chiffrée (ex. temps de réaction en ms) et un état sûr défini.,"Réponse simulée (mode démo), construite à partir des règles automatiques.",À VALIDER


## Exporter pour revue

In [7]:
from safety_agent.checks import run_all_checks
from safety_agent.report import markdown_report

out = ROOT / "rapport_relecture.md"
out.write_text(markdown_report(project, run_all_checks(project) + ai_findings, proposals,
                               "Relecture des exigences"), encoding="utf-8")
print("Rapport écrit :", out.relative_to(ROOT))

Rapport écrit : rapport_relecture.md
